# 00 — Materialize the hybrid *E. coli*-related benchmark
Reuses the shared 15 Gbp Drive corpus. It creates guarded within-genome train/validation/test coordinates plus ANI-99 held-out validation and test genomes. Run on a CPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/ATCG-FM || git clone https://github.com/DRAGGON-Lab/ATCG-FM.git /content/ATCG-FM
%pip install -q "gdown>=5,<6" "pandas>=2,<3" "pyarrow>=21,<26"
%pip install -e /content/ATCG-FM/packages/atcg-sequence


In [ ]:
from pathlib import Path
from collections import defaultdict
import gzip, hashlib, json, re
import gdown, pandas as pd
from atcg.sequence import FixedAlphabetTokenizer, SequenceRecord, iter_fasta, prepare_coordinate_splits, reverse_complement, write_fasta_lines

SOURCE_FOLDER = '1b_bLHFGeQP7RLhOmE8-UIkT4gEFRB2QW'
ROOT = Path('/content/drive/MyDrive/ATCG-FM/ecoli_hybrid_v1')
STAGE = ROOT / 'stage-small'; WORK = Path('/content/ecoli_hybrid_build')
STAGE.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
SEED = 17; GENOMES_PER_ROLE = 16; STREAM_LENGTH = 16_384; MINIMUM_STREAM = 129; GUARD_BASES = 128
WITHIN_USABLE = 1_250_000; WITHIN_SOURCE = WITHIN_USABLE + 2 * GUARD_BASES; HELDOUT_BASES = 125_000
MANIFEST_IDS = {'accessions':'1JgJBOz0msdBZrU0KGmYMdgJfByxeSHVg', 'shards':'1kH4luK-tZWzmgYKaebAgpoiqzx0238Y1'}
SHARD_IDS = {
 'train': ['1eUVgYluKWnBexh9g4dxKpRA3qhN9cv33','1PtDceepdNx-nPdd-yIXFO6WmV2ewhBLM','1r3QsnVm17iqUWbcKDDHivGDRcSNzHCd7','1ndr9KDju435E_xcRMkPbhGA4_LhGpInt','11uWygTsxI_6SATmio58U9cKWIUuxpBBd','1TSf7kkwXwEoc1BMCkRQeyJSjfJeb74qT','1xuEnd2A4LNdS8yBtzq_rlyYbRykGtgDe','1tlRLuEJ6Y9wTB4rWTfryh6d2TPMV4uEb','1QJiRAILOApf6vd2fPkIH-oGS4w1436a-','10GsgOy_NY_4P6w3zLfkd49cNyLjxfAZd','1Q0Vwi0JPauT22TzI12IQwhDHRJgw3aBB','1_b-ar3-R8rG9zKY4JYAzRqs8WRtkTJpR','1N9fyNVn8hgYJRp1kaDQpMQzBLEmq5ycT','1Xk3EZgdsPhAkyqtr-2ojFeetlgap1GE6','1ur_DJiINeyuX9o3iTOXkTi8EIb3lxZy1','1PIsQWMjpyeJWW1Uulw892iLAfICqUu3t','1Fkqi8Ug4k6a-qGM3KeJ-pZ0VxY8DlFkf','1t_0t7Q5GgGs7VNbiOlvGwNAI781cbzbU','1FNuIOPFG98BukT9pxbXQIS1WGunYlp1_','1_c6Cdk6yRKa1BRSx6dXFbRloHBLlqXod','1wyHXJ19JGXEhMmiuoipjFoPaVBfI6ahZ','1JATH3-Jr2wLuIoUM-lVZew19853lY6Q4','1ifnqsO6koCKI6I1i6T6zxr4vAb0uNjuA','16sqigX0TviyNbLOV2at2NveUv5SqtKeP','1Z0iC8hHXsjAaL29GiROPvacW8eCqOOcZ','1VkE25r4tY3J-CJkLFZ1ClMlaygmQZbNQ'],
 'val': ['1BCv8VtD6hk54ZtnC1YYNYBLlowJlQJcU','1VKyq7sU5R3pECanfKWUg6JSOneujUvE7'],
 'test': ['1TaNHCuVEWdEGDZqe-AMk7KgspeOD3a-4','10xl4_7z4YOcSpcnAxBI4jtUjt9uaMizI','1yxb5uKStxGm7nBbs4y7wiY1J5rV6Z51O']
}

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(1024 * 1024): digest.update(chunk)
    return digest.hexdigest()

def download(file_id, path):
    if not path.exists(): gdown.download(id=file_id, output=str(path), quiet=False)
    return path

accession_path = download(MANIFEST_IDS['accessions'], WORK / 'accession_manifest.csv')
shard_manifest_path = download(MANIFEST_IDS['shards'], WORK / 'fasta_shard_manifest.csv')
metadata = pd.read_csv(accession_path, low_memory=False); shard_manifest = pd.read_csv(shard_manifest_path)
print(len(metadata), 'accessions', shard_manifest.groupby('split').bases.sum().to_dict())


In [ ]:
ACCESSION_RE = re.compile(r'(?:RS_|GB_)?(GC[AF]_\d+\.\d+)')
def accession_from_header(identifier, description):
    match = ACCESSION_RE.search(identifier + ' ' + description)
    return match.group(1) if match else None
def accession_from_record(record): return accession_from_header(record.identifier, record.description)

def eligible_rows(split, minimum_longest):
    rows = metadata[(metadata.split == split) & (metadata.completeness >= 95) & (metadata.contamination <= 5) & (metadata.longest_contig >= minimum_longest)].copy()
    rows['rank_value'] = pd.to_numeric(rows.selection_rank, errors='coerce').fillna(10**12)
    return rows.sort_values(['rank_value','quality_score','accession'], ascending=[True,False,True])

def choose_distinct(found, rows):
    chosen = []; groups = set()
    by_accession = rows.set_index('accession', drop=False)
    for accession in rows.accession:
        if accession not in found: continue
        group = str(by_accession.loc[accession].clade_group)
        if group in groups: continue
        chosen.append(accession); groups.add(group)
        if len(chosen) == GENOMES_PER_ROLE: break
    return chosen

def collect_genomes(split, minimum_longest):
    rows = eligible_rows(split, minimum_longest); candidates = set(rows.accession); found = {} ; used = []
    for shard_index, file_id in enumerate(SHARD_IDS[split]):
        path = download(file_id, WORK / f'{split}_shard_{shard_index:05d}.fa.gz')
        expected_row = shard_manifest[(shard_manifest.split == split) & (shard_manifest.shard_index == shard_index)]
        if len(expected_row) != 1 or sha256(path) != expected_row.iloc[0].sha256: raise RuntimeError(f'{split} shard {shard_index} checksum mismatch')
        used.append({'split':split, 'shard_index':shard_index, 'file_id':file_id, 'sha256':sha256(path)})
        group_by_accession = rows.set_index('accession').clade_group.astype(str).to_dict(); found_groups = set()
        with gzip.open(path, 'rt') as handle:
            selected_records = iter_fasta(handle, source=str(path), record_filter=lambda identifier, description: accession_from_header(identifier, description) in candidates)
            for record in selected_records:
                accession = accession_from_record(record)
                if accession not in candidates or len(record.sequence) < minimum_longest: continue
                if accession not in found or len(record.sequence) > len(found[accession].sequence): found[accession] = record
                found_groups.add(group_by_accession[accession])
                if len(found_groups) >= 24: break
        chosen = choose_distinct(found, rows)
        print(split, 'after shard', shard_index, 'eligible distinct genomes:', len(chosen))
        if len(chosen) == GENOMES_PER_ROLE: return chosen, found, rows.set_index('accession', drop=False), used
    raise RuntimeError(f'{split}: found only {len(choose_distinct(found, rows))} eligible distinct clades; add shard IDs or inspect FASTA accession headers')

train_accessions, train_found, train_rows, train_sources = collect_genomes('train', WITHIN_SOURCE)
val_accessions, val_found, val_rows, val_sources = collect_genomes('val', HELDOUT_BASES)
test_accessions, test_found, test_rows, test_sources = collect_genomes('test', HELDOUT_BASES)
print({'train':train_accessions, 'validation_clade':val_accessions, 'test_clade':test_accessions})


In [ ]:
def deterministic_window(accession, record, length, role):
    available = len(record.sequence) - length
    offset = int(hashlib.sha256(f'{SEED}:{role}:{accession}'.encode()).hexdigest()[:16], 16) % (available + 1)
    source_id = f'{accession}__{record.identifier}__source_{offset}_{offset+length}'
    value = SequenceRecord(identifier=source_id, sequence=record.sequence[offset:offset+length].upper(), metadata={'accession':accession,'contig':record.identifier,'source_start':str(offset),'source_stop':str(offset+length)})
    return value, offset

def globalize(records, source_offset, role):
    output = []
    for record in records:
        local_start = int(record.metadata['start']); local_stop = int(record.metadata['stop'])
        start = source_offset + local_start; stop = source_offset + local_stop
        output.append(SequenceRecord(identifier=f"{record.metadata['accession']}__{record.metadata['contig']}__{role}__{start}_{stop}", sequence=record.sequence, metadata={**record.metadata,'role':role,'source_start':str(start),'source_stop':str(stop)}))
    return output

splits = defaultdict(list); coordinate_contract = []
for accession in train_accessions:
    source, source_offset = deterministic_window(accession, train_found[accession], WITHIN_SOURCE, 'within')
    prepared = prepare_coordinate_splits([source], stream_length=STREAM_LENGTH, minimum_stream_length=MINIMUM_STREAM, guard_bases=GUARD_BASES)
    for split, output_name in [('train','train'),('validation','validation_within'),('test','test_within')]:
        splits[output_name].extend(globalize(prepared.records(split), source_offset, output_name))
    coordinate_contract.append({'accession':accession,'contig':train_found[accession].identifier,'window_start':source_offset,'window_stop':source_offset+WITHIN_SOURCE,'train':[source_offset,source_offset+1_000_000],'validation_within':[source_offset+1_000_000+GUARD_BASES,source_offset+1_125_000+GUARD_BASES],'test_within':[source_offset+1_125_000+2*GUARD_BASES,source_offset+WITHIN_SOURCE]})

def heldout_streams(accessions, found, role):
    output = []
    for accession in accessions:
        source, source_offset = deterministic_window(accession, found[accession], HELDOUT_BASES, role)
        for local_start in range(0, len(source.sequence), STREAM_LENGTH):
            local_stop = min(local_start + STREAM_LENGTH, len(source.sequence))
            if local_stop - local_start < MINIMUM_STREAM: continue
            start = source_offset + local_start; stop = source_offset + local_stop
            output.append(SequenceRecord(identifier=f'{accession}__{found[accession].identifier}__{role}__{start}_{stop}', sequence=source.sequence[local_start:local_stop], metadata={'accession':accession,'contig':found[accession].identifier,'role':role,'source_start':str(start),'source_stop':str(stop)}))
    return output
splits['validation_clade'] = heldout_streams(val_accessions, val_found, 'validation_clade')
splits['test_clade'] = heldout_streams(test_accessions, test_found, 'test_clade')

allowed = set(FixedAlphabetTokenizer().alphabet)
invalid = {record.identifier:sorted(set(record.sequence)-allowed) for records in splits.values() for record in records if set(record.sequence)-allowed}
if invalid: raise RuntimeError(f'sequences violate tokenizer alphabet: {list(invalid.items())[:5]}')
seen = {}
for split in ('train','validation_within','test_within','validation_clade','test_clade'):
    for record in splits[split]:
        canonical = min(record.sequence, reverse_complement(record.sequence))
        digest = hashlib.sha256(canonical.encode()).hexdigest()
        if digest in seen and seen[digest] != split: raise RuntimeError(f'exact canonical duplicate crosses {seen[digest]} and {split}: {record.identifier}')
        seen.setdefault(digest, split)

base_counts = {name:sum(len(record.sequence) for record in records) for name, records in splits.items()}
expected_counts = {'train':16_000_000,'validation_within':2_000_000,'test_within':2_000_000,'validation_clade':2_000_000,'test_clade':2_000_000}
if base_counts != expected_counts: raise RuntimeError(f'base quotas differ: {base_counts}')
print(base_counts)


In [ ]:
split_sha256 = {}; fingerprint = hashlib.sha256()
for split in ('train','validation_within','test_within','validation_clade','test_clade'):
    path = STAGE / f'{split}.fa.gz'
    with gzip.open(path, 'wt') as handle: handle.writelines(write_fasta_lines(splits[split]))
    split_sha256[split] = sha256(path)
    fingerprint.update(split.encode()); fingerprint.update(bytes.fromhex(split_sha256[split]))

def role_metadata(accessions, rows):
    return [{'accession':a,'clade_group':str(rows.loc[a].clade_group),'quality_score':float(rows.loc[a].quality_score),'completeness':float(rows.loc[a].completeness),'contamination':float(rows.loc[a].contamination)} for a in accessions]
roles = {'within':role_metadata(train_accessions, train_rows),'validation_clade':role_metadata(val_accessions, val_rows),'test_clade':role_metadata(test_accessions, test_rows)}
train_groups = {row['clade_group'] for row in roles['within']}; val_groups = {row['clade_group'] for row in roles['validation_clade']}; test_groups = {row['clade_group'] for row in roles['test_clade']}
if train_groups & val_groups or train_groups & test_groups or val_groups & test_groups: raise RuntimeError('ANI clade leakage detected')
manifest = {'schema_version':2,'dataset_id':'ecoli-hybrid-v1-stage-small','source_dataset':'bacteria_titan_v1_ecoli_related_15gbp','source_folder_id':SOURCE_FOLDER,'seed':SEED,'stream_length':STREAM_LENGTH,'minimum_stream_length':MINIMUM_STREAM,'guard_bases':GUARD_BASES,'tokenizer_contract':'FixedAlphabetTokenizer IUPAC; BOS included, EOS excluded; one target per base; no reverse-complement augmentation','split_policy':{'within':'per-genome contiguous 80/10/10 with 128-base guards','clade':'inherited ANI-99 clade-group holdout'},'roles':roles,'coordinate_contract':coordinate_contract,'base_counts':base_counts,'stream_counts':{name:len(records) for name,records in splits.items()},'split_sha256':split_sha256,'dataset_fingerprint':fingerprint.hexdigest(),'source_files':train_sources+val_sources+test_sources,'source_manifest_sha256':{'accessions':sha256(accession_path),'shards':sha256(shard_manifest_path)},'duplicate_policy':'canonical forward/reverse-complement SHA-256; cross-partition duplicates rejected'}
(STAGE / 'manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
print('saved hybrid benchmark to', STAGE, manifest['dataset_fingerprint'])
